# Où naît le flou ? — exécution sur Colab (GPU T4)

Ce notebook enchaîne les étapes du projet. **Aucune cellule n'a été exécutée par l'auteur** : le code PyTorch du dépôt n'a jamais
tourné là où il a été écrit (pas de PyTorch). La cellule 3 (« test de fumée ») est donc obligatoire avant tout entraînement.

Avant de commencer : *Exécution → Modifier le type d'exécution → GPU T4*. Une session gratuite peut s'interrompre : tout est
**reprenable** (relancer la même cellule). Les résultats sont écrits sur Google Drive pour survivre aux coupures.

**Interface web ou VS Code ?** Les deux pilotent le même runtime et les cellules fonctionnent dans les deux. Le seul point à vérifier est **Google Drive** : son montage demande une autorisation à la page du notebook et peut échouer depuis VS Code (`MessageError: [dfs_ephemeral] ... unauthorized`). La cellule 2 le détecte et affiche `persistant : True` ou `False`. Si c'est `False`, les `runs/` vivent sur le disque de la session et disparaissent avec elle : lancer alors la campagne (étapes 7 et 9) depuis le navigateur. Si c'est `True`, VS Code convient pour tout.


## 1. GPU, dépôt et Drive

In [3]:
!nvidia-smi -L || echo "AUCUN GPU : Exécution -> Modifier le type d'exécution -> T4 (obligatoire pour les étapes 3, 7 et 9)"

# Le montage de Drive demande une autorisation à la page du notebook : il peut échouer depuis VS Code
# (erreur « dfs_ephemeral ... unauthorized »). On bascule alors sur le disque de la session, en le disant.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, PERSISTANT = '/content/drive/MyDrive/flou_wm', True
except Exception as e:
    BASE, PERSISTANT = '/content/flou_wm_resultats', False
    print(f"Drive indisponible ({type(e).__name__}). Les résultats iront dans {BASE},")
    print("qui DISPARAÎT à la fin de la session : entraînements non reprenables.")
    print("Pour la campagne complète, ouvrir ce notebook dans l'interface web de Colab.")
print(f"résultats -> {BASE}   (persistant : {PERSISTANT})")

Mounted at /content/drive
résultats -> /content/drive/MyDrive/flou_wm   (persistant : True)


In [ ]:
# Cloner le dépôt (dépôt privé : mettre un jeton -> https://<TOKEN>@github.com/...)
import os
REPO = 'https://github.com/CedricDada/flou-world-models.git'
if os.path.isdir('/content/flou_wm/.git'):
    # session reprise : récupérer les correctifs ; échoue si data/ runs/ reports/ sont déjà des liens
    !cd /content/flou_wm && git pull -q --ff-only || echo "git pull ignoré : supprimer /content/flou_wm pour repartir d'un clone neuf"
else:
    !git clone -q $REPO /content/flou_wm
%cd /content/flou_wm

# runs/ reports/ data/ pointent vers BASE (Drive si disponible) : reprise possible après une coupure
import shutil
for d in ('runs', 'reports', 'data'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
    if os.path.isdir(d) and not os.path.islink(d):
        shutil.rmtree(d)
    if not os.path.islink(d):
        os.symlink(f'{BASE}/{d}', d)

# Colab fournit déjà numpy, scipy, matplotlib et pillow : n'installer que ce qui manque,
# pour ne pas risquer de remplacer la numpy sur laquelle le PyTorch préinstallé est compilé.
import importlib.util
manquants = [pkg for mod, pkg in (('numpy', 'numpy'), ('scipy', 'scipy'), ('matplotlib', 'matplotlib'),
                                  ('PIL', 'pillow'), ('pytest', 'pytest'))
             if importlib.util.find_spec(mod) is None]
if manquants:
    !pip install -q {' '.join(manquants)}
print('paquets installés :', ' '.join(manquants) if manquants else 'aucun, tout était présent')

## 2. Tests NumPy (jeu, bots, instruments de mesure, analyse)

Ils ne demandent pas PyTorch. Une cellule suffit ; attendre environ 2 à 4 minutes.

In [ ]:
!python -m pytest -q -x

## 3. Test de fumée du code PyTorch (obligatoire, une fois)

Vérifie les nombres de paramètres, l'entraînement, la reprise après interruption et l'évaluation de chaque famille sur de minuscules
données ; avec `--time`, mesure aussi le temps par pas à la taille réelle (écrit `reports/timing.json`). Si une ligne est `[FAIL]`,
**ne pas lancer le plan** : copier la trace d'erreur et la corriger d'abord.

In [ ]:
!python scripts/smoke_test.py --time

## 4. Données

Les valeurs de p du plan (0,1 / 0,3 / 0,5) plus p = 0 (contrôles), en entraînement, validation et test. Générées une fois, sur Drive.

In [ ]:
!python scripts/generate_data.py --split train --p 0.0 0.1 0.3 0.5 --n-transitions 100000
!python scripts/generate_data.py --split val   --p 0.0 0.1 0.3 0.5 --n-transitions 20000
!python scripts/generate_data.py --split test  --p 0.0 0.1 0.3 0.5 --n-bifurcations 1000

## 5. Instrument de mesure (avant tout modèle)

Règle ρmax et contrôle H0a sans entraînement. Si H0a échoue, **ne mesurer aucun modèle**.

In [ ]:
!python scripts/validate_instrument.py

## 6. Plan et budget

Le nombre de pas par run se règle sur le temps **mesuré** à l'étape 3 (pas sur une valeur supposée).

In [ ]:
!python scripts/plan.py --timing reports/timing.json
# Pour caler le plan sur un budget : par exemple 38 h de GPU
!python scripts/plan.py --timing reports/timing.json --budget-hours 38

## 7. Exécution

Cœur et contrôles d'abord. Relancer la même cellule après une coupure : elle reprend. Remplacer `--steps` par la valeur de l'étape 6.
Une session T4 gratuite dure rarement plus de quelques heures : `--max-hours` arrête proprement avant la coupure.

`--max-hours 10` n'est plus là pour la coupure des sessions gratuites (Colab Pro va jusqu'à 24 h) mais comme **garde-fou de consommation** : l'exécution s'arrête proprement, écrit `runs/plan_status.json`, et il suffit de relancer la même cellule pour continuer. Surveiller le compteur d'unités de calcul entre deux relances.


In [ ]:
!python scripts/run_plan.py --steps 20000 --max-hours 10 --groups core control

In [ ]:
!python scripts/run_plan.py --steps 20000 --max-hours 10 --skip-groups core control p_axis

In [ ]:
!python scripts/run_plan.py --steps 20000 --max-hours 10 --groups p_axis    # exploratoire, sacrifié en premier

## 8. Analyse

Verdicts H0 à H3 (H4 et H5 restent « non évaluables » tant que l'étape 9 n'est pas faite), tables, figures, rapport. Puis, à la main : la relecture de 100 cas et le test A/B (voir `docs/EXPERIMENTS.md`).

In [ ]:
!python scripts/analyze.py

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('reports/report.md', encoding='utf-8').read()))
for f in ('fig_flou.png', 'fig_calibration.png', 'fig_rappel.png'):
    if os.path.exists(f'reports/{f}'): display(Image(f'reports/{f}'))

## 9. Extension « décision » (facultatif) : le flou change-t-il une décision ?

Hypothèses H4 et H5, guidées par PETS (arXiv:1805.12114), PlaNet (arXiv:1811.04551) et Objective Mismatch (arXiv:2002.04523) : voir `docs/EXTENSION_DECISION.md`.
D'abord les contrôles sans modèle (H0e, H0f ; quelques minutes de CPU), puis le niveau 1, rapide.

In [ ]:
!python scripts/validate_decision.py
!python scripts/evaluate_decision.py --all

Niveau 2 (planification dans le modèle) : **long**. La première cellule chronomètre sur 4 parties de 10 pas ; l'étape 3 (`smoke_test.py --time`) a aussi affiché une estimation.
Si c'est trop long, réduire `--games` ou `--candidates` **pour tous les runs**. Relancer la même cellule après une coupure : les runs déjà faits sont sautés.

In [ ]:
!python scripts/evaluate_decision.py --all --level 2 --games 4 --game-steps 10 --no-save   # chronométrage seulement, n'écrit rien

In [ ]:
!python scripts/evaluate_decision.py --all --level 2   # mesure complète ; reprenable

In [ ]:
!python scripts/analyze.py   # le rapport contient maintenant H4 et H5